# Schrödinger picture kinetic momentun broadening calculation

We try here to keep the exact approximation between Milne and light-cone coordinates. To deal with the point outside of the Glasma light cone that might be proved by the finite size of the wavepackage we approximate the value of the fields at $z_i > x_i^+$ by the value at $z_i = x_i^+$

### Set up

Set the parameters and environment variables

In [1]:
import numpy as np

# hbar * c [GeV * fm]
hbarc = 0.197326 

# Simulation box 
L = 5         
N = 256
tau_sim = 0.3125    
DTS = 16

# Derived parameters
a = L/N
E0 = N / L * hbarc
DT = 1.0 / DTS
maxt = int(tau_sim / a * DTS)
nplus = maxt//DTS

# Glasma fields
su_group = 'su3'
uv = 10.0
ir = 0.2
g2mu = 1.5


g = 2.0          		
mu = g2mu / g**2

ns = 50      

nevents = 10


In [2]:
import os

os.environ["MY_NUMBA_TARGET"] = "cuda"
os.environ["PRECISION"] = "double"
os.environ['GAUGE_GROUP'] = su_group

# Import relevant modules
import sys
sys.path.append('..')

# Glasma modules
import curraun.core as core
import curraun.mv as mv
import curraun.initial as initial
initial.DEBUG = False

import curraun.su as su
from curraun.numba_target import use_cuda
if use_cuda:
    from numba import cuda

import curraun.su as su
import curraun.pi_exact_fields as pi

/home2/carlos.lamas/condacurraun/lib/python3.10/site-packages/scipy/__init__.py:132: UserWarning: A NumPy version >=1.21.6 and <1.28.0 is required for this version of SciPy (detected version 1.21.5)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Using CUDA
Using SU(3)
Using double precision


### Simulation

We define the simulation routine

In [3]:
from tqdm import tqdm

# Simulation rutine
def simulate():
    output = {}
    
    # Derived parameters
    a = L/N
    E0 = N / L * hbarc
    DT = 1.0 / DTS
    maxt = int(tau_sim / a * DTS)
    nplus = maxt//DTS
    
    # We create the object simulation
    s = core.Simulation(N, DT, g)

    # We initilize the Glasma fields
    va = mv.wilson(s, mu=mu / E0, m=ir / E0, uv=uv / E0, num_sheets=ns)
    vb = mv.wilson(s, mu=mu / E0, m=ir / E0, uv=uv / E0, num_sheets=ns)
    initial.init(s, va, vb)
    
    # We create objects to store the Glasma fields at every time step
    ux = np.zeros((maxt, N*N, su.GROUP_ELEMENTS), dtype=su.GROUP_TYPE)
    uy = np.zeros((maxt, N*N, su.GROUP_ELEMENTS), dtype=su.GROUP_TYPE)
    Aeta = np.zeros((maxt, N*N, su.GROUP_ELEMENTS), dtype=su.GROUP_TYPE)
    
    # We simulate the event and store the fields
    with tqdm(total=maxt) as pbar:    
        for t in range(maxt):
            
            # Evolve the Glasma fields
            core.evolve_leapfrog(s)
                
            u1 = s.u1.copy()
            ux[t] = u1[:, 0, :]
            uy[t] = u1[:, 1, :]
            
            Aeta[t] = s.aeta1.copy()
                
                
            pbar.update(1) 
    
    # We free the memory 
    if use_cuda:
        cuda.current_context().deallocations.clear()
    
    # We reshape the fields to make them efficient on the GPU
    ux = ux.reshape(maxt*N*N, su.GROUP_ELEMENTS)
    uy = uy.reshape(maxt*N*N, su.GROUP_ELEMENTS)
    Aeta = Aeta.reshape(maxt*N*N, su.GROUP_ELEMENTS)
    
    # We create objects to store the jet evolution fields
    up = np.zeros((nplus, N*N, su.GROUP_ELEMENTS), dtype=su.GROUP_TYPE)
    Ay = np.zeros((nplus, N*N, su.GROUP_ELEMENTS), dtype=su.GROUP_TYPE)
    Az = np.zeros((nplus, N*N, su.GROUP_ELEMENTS), dtype=su.GROUP_TYPE)
    
    # We create the object Glasma fields
    gf = pi.GlasmaFields(s, ux, uy, Aeta, DTS)
    gf.init()
    
    # We simulate extract the value of this fields when the spatial and temporal lattices match
    with tqdm(total=nplus) as pbar:    
        for xplus in range(nplus):
                
            gf.compute_fields(xplus, a/hbarc)
            
            if xplus != 0:
            
                up[xplus-1] = gf.up.copy()
                Ay[xplus-1] = gf.ay.copy()
                Az[xplus-1] = gf.az.copy()
                
            pbar.update(1) 
    
    if use_cuda:
        cuda.current_context().deallocations.clear()
    
    
    # We write the transformed fields in a dictionary
    output["up"] = up
    output["Ay"] = Ay
    output["Az"] = Az
    
    return output

We run the simulation

In [4]:
import warnings
warnings.filterwarnings('ignore')

for n in range (nevents):
    
    print('Event %i' % n)
    output = simulate()
    
    up = output['up']
    Ay = output['Ay']
    Az = output['Az']

    # Save the files
    save_dir = os.path.join('..', 'simulations', 'test_exact_L=5_N=256_t0')
    os.makedirs(save_dir, exist_ok=True)

    np.save(os.path.join(save_dir, 'up_%i.npy' % n), up)
    np.save(os.path.join(save_dir, 'Ay_%i.npy' % n), Ay)
    np.save(os.path.join(save_dir, 'Az_%i.npy' % n), Az)

Event 0


100%|██████████| 16/16 [00:00<00:00, 22.85it/s]


Event 1


100%|██████████| 16/16 [00:00<00:00, 102.37it/s]


Event 2


...

100%|██████████| 16/16 [00:00<00:00, 104.79it/s]


Event 8


100%|██████████| 16/16 [00:00<00:00, 106.34it/s]


Event 9


100%|██████████| 16/16 [00:00<00:00, 106.31it/s]
